# Importing Libraries

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import random
import optuna
import evaluate

from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)
from datasets import Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Importing datasets

In [4]:
! gdown 1o_E9XCDt0iMkcJhvwQbRrVpfo3FAi2nq
! gdown 1U_NPI_FyBOMSNtcJEBNONDeYoOLMNIrY
! gdown 11uurHsNZAjG70Ooq0noI7a2W4rbiNvIw

Downloading...
From: https://drive.google.com/uc?id=1o_E9XCDt0iMkcJhvwQbRrVpfo3FAi2nq
To: /content/jigsaw_bert_dataset.csv
100% 20.9M/20.9M [00:00<00:00, 66.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U_NPI_FyBOMSNtcJEBNONDeYoOLMNIrY
To: /content/twitter_bert_dataset.csv
100% 1.15M/1.15M [00:00<00:00, 11.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=11uurHsNZAjG70Ooq0noI7a2W4rbiNvIw
To: /content/combined_model_dataset.csv
100% 22.0M/22.0M [00:00<00:00, 78.0MB/s]


In [5]:
df_google = pd.read_csv('/content/jigsaw_bert_dataset.csv')
df_twitter = pd.read_csv('/content/twitter_bert_dataset.csv')
df_combined = pd.read_csv('/content/combined_model_dataset.csv')


# Data preperation

## Text preprocessing

In [6]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 15.5 MB/s eta 0:00:00


In [7]:
import re
import nltk
import contractions
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))- {"not", "no"}

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
def preprocess_bert(text):
  if not text or str(text).strip() == "":
    return ""
  # lowercase
  text = text.lower()

  # remove mentions
  text = re.sub(r'@\w+', 'user', text)

  # remove hashtag symbol but keep word
  text = re.sub(r'#', '', text)

  # remove URLs
  text = re.sub(r'http\S+|www\S+', '', text)

  # remove black bars
  text = re.sub(r'█+', ' ', text)

  # remove extra spaces
  text = re.sub(r'\s+', ' ', text).strip()

  return text

In [9]:
df_google["text"] = df_google["text"].apply(preprocess_bert)
df_google

,text,label
0,"was the girl on any prescription drugs? if so,...",0
1,"while the tax will have an effect, the real ef...",0
2,what makes you think i believe the posts in qu...,0
3,and so canada devolves into nonfunctional irre...,0
4,"boomer 88, a rather defeatist attitude and lam...",0
...,...,...
69995,and another childish insult from clearly an em...,1
69996,"no, you just call them deviants and perverts.",1
69997,"well said, bumpkin. i'm more inclined to liste...",1
69998,did i mention that mars is undergoing climate ...,1


In [10]:
df_twitter["text"] = df_twitter["text"].apply(preprocess_bert)
df_twitter

,text,label
0,user nice new signage. are you not concerned b...,0
1,a woman who you fucked multiple times saying y...,1
2,user user real talk do you have eyes or were t...,1
3,your girlfriend lookin at me like a groupie in...,1
4,hysterical woman like user,0
...,...,...
8949,oooohhhh bitch didn't even listen to the dead ...,0
8950,user good luck user more americans walkawayfro...,0
8951,bitch you can't keep up so stop trying,1
8952,user user user user user user japan is always ...,0


In [11]:
df_combined["text"] = df_combined["text"].apply(preprocess_bert)
df_combined

,text,label
0,"this is politicians' strategy ""action through ...",0
1,he has underlings to do tjat for him,0
2,"judge on separation of immigrant families: ""if...",0
3,to the republican nra(nazi's razing america)th...,0
4,somebody musta complained.,0
...,...,...
78949,throw a little pesticide on top of carcinogen ...,0
78950,this man is a coward & an oath breaker & the w...,1
78951,a propublica investigation found that police r...,0
78952,you answered the question better than i could ...,0


## Spliting dataset into train and test

## Splitting combined dataset

In [12]:
X = df_combined['text']
y = df_combined['label']

In [13]:
X_tr, X_test, y_tr, y_test = train_test_split(X,y,test_size = 0.20, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_tr,y_tr,test_size = 0.25, random_state = 42, stratify = y_tr)

In [14]:
X_train.shape

(47372,)

In [15]:
X_test.shape

(15791,)

## Splitting google, twitter and reddit datasets

In [16]:
google_train_df,google_val_df = train_test_split(
    df_google,
    test_size = 0.2,
    random_state = SEED,
    stratify = df_google['label']
)

In [17]:
twitter_train_df, twitter_temp_df = train_test_split(
    df_twitter,
    test_size = 0.4,
    random_state = SEED,
    stratify = df_twitter['label']
)

twitter_val_df, twitter_test_df = train_test_split(
    twitter_temp_df,
    test_size = 0.5,
    random_state = SEED,
    stratify = twitter_temp_df['label']
)

In [18]:
print("Google train:", google_train_df.shape)
print("Google val:", google_val_df.shape)
print("Twitter train:", twitter_train_df.shape)
print("Twitter val:", twitter_val_df.shape)
print("Twitter test:", twitter_test_df.shape)

Google train: (56000, 2)
Google val: (14000, 2)
Twitter train: (5372, 2)
Twitter val: (1791, 2)
Twitter test: (1791, 2)


##  Converting to Hugging Face datasets

In [19]:
google_train_ds = Dataset.from_pandas(google_train_df.reset_index(drop=True))
google_val_ds = Dataset.from_pandas(google_val_df.reset_index(drop=True))
google_train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 56000
})

In [20]:
twitter_train_ds = Dataset.from_pandas(twitter_train_df.reset_index(drop=True))
twitter_val_ds = Dataset.from_pandas(twitter_val_df.reset_index(drop=True))
twitter_test_ds = Dataset.from_pandas(twitter_test_df.reset_index(drop=True))
twitter_train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 5372
})

## Tokenization

In [21]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
MAX_LENGTH = 128
def tokenize_batch(batch):
  return tokenizer(
      batch['text'],
      truncation = True,
      max_length = MAX_LENGTH
  )

In [23]:
google_train_tok = google_train_ds.map(tokenize_batch, batched = True)
google_val_tok = google_val_ds.map(tokenize_batch, batched = True)


In [24]:
twitter_train_tok = twitter_train_ds.map(tokenize_batch, batched = True)
twitter_val_tok = twitter_val_ds.map(tokenize_batch, batched = True)
twitter_test_tok = twitter_test_ds.map(tokenize_batch, batched = True)


In [25]:
twitter_test_tok

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1791
})

## Set dataset format

In [26]:
cols = ['input_ids','attention_mask','label']
google_train_tok.set_format(type = 'torch', columns = cols)
google_val_tok.set_format(type = 'torch', columns = cols)

In [27]:
twitter_train_tok.set_format(type = 'torch', columns = cols)
twitter_val_tok.set_format(type = 'torch', columns = cols)
twitter_test_tok.set_format(type = 'torch', columns = cols)

# Evaluation metrics

In [28]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }

# Hyperparameter tuning

In [29]:
def build_model():
  return AutoModelForSequenceClassification.from_pretrained(
      MODEL_NAME,
      num_labels = 2
  )
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [30]:
def objective(trial):
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
  batch_size = trial.suggest_categorical("batch_size",[8,16])
  num_train_epochs = trial.suggest_int("num_train_epochs",1,2)
  weight_decay = trial.suggest_float("weight_decay", 0.0, 1.0)
  model = build_model()
  args = TrainingArguments(
      output_dir = f"./stage1_trial_{trial.number}",
      eval_strategy = "epoch",
      save_strategy = 'epoch',
      logging_strategy = "epoch",
      learning_rate = learning_rate,
      per_device_train_batch_size = batch_size,
      per_device_eval_batch_size = batch_size,
      weight_decay = weight_decay,
      load_best_model_at_end = True,
      metric_for_best_model = 'f1',
      greater_is_better = True,
      report_to = "none",
      seed = SEED,
      fp16=torch.cuda.is_available()
  )
  trainer = Trainer(
      model = model,
      args = args,
      train_dataset = google_train_tok,
      eval_dataset = google_val_tok,
      data_collator = data_collator,
      compute_metrics = compute_metrics,
      callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]

  )
  trainer.train()
  eval_results = trainer.evaluate()
  return eval_results["eval_f1"]

In [ ]:
study = optuna.create_study(direction = "maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials = 5)
print("Best params:", study.best_params)
print("Best F1:", study.best_value)

[I 2026-04-05 23:28:18,669] A new study created in memory with name: no-name-98fcbaaa-c6fe-4000-8882-fec78dcb9eef


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.350472,0.304322,0.887571,0.883192,0.893286,0.888210
2,0.263425,0.339172,0.886643,0.879009,0.896714,0.887773


...

[I 2026-04-05 23:37:43,528] Trial 0 finished with value: 0.8884150675195451 and parameters: {'learning_rate': 1.827226177606625e-05, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.15601864044243652}. Best is trial 0 with value: 0.8884150675195451.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.325881,0.279130,0.890071,0.876673,0.907857,0.891992


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]